# Median vs. VAE imputation for VaDeSC on SUPPORT

Follow-up to `05_vae_vadesc_on_VAE_imputed_support.ipynb`, with two fixes:

1. **Separate, correctly labelled comparisons.** In notebook 05 the "Feature leakage" table compared VAE-imputed with median-imputed data (both without the post-baseline variables), not leakage. Here the imputation comparison (section 1) and the leakage comparison (section 3) are separate.
2. **Several data splits and seeds.** Notebook 05 compared one VaDeSC run per imputation. Here every comparison uses 3 train/val/test splits and 5 VaDeSC seeds, the VAE is re-trained on the training rows of each split, and everything runs on the same machine. VaDeSC runs are paired (same split, same seed) for the two imputations.

The VAE is the one from notebook 04 (Encoder Indicators), packaged in `imputation/vae_impute.py`. It never sees survival times or events.

Full run: about 1-1.5 h on a CPU. `QUICK = True` for a short check.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy import stats

from imputation.vae_impute import load_support_from_frame, vae_impute_support
from src.baselines import run_cox, run_rsf
from src.data import load_support
from src.vadesc import DEVICE, evaluate_vadesc, train_vadesc

QUICK = False
SPLITS = [42, 1] if QUICK else [42, 1, 2]           # 42 = the split used in notebooks 01, 04 and 05
VADESC_SEEDS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
LEAK_SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 60 if QUICK else 300
print("device:", DEVICE)

device: cpu


## 1. Imputation comparison (without post-baseline variables)

For each split: median/mode imputation (`load_support`) vs. VAE imputation, then Cox, RSF and VaDeSC on identical preprocessing.

In [2]:
def run_models(d, seeds, split, label):
    rows = [{"split": split, "data": label, "model": "Cox PH", "seed": None, **run_cox(d)},
            {"split": split, "data": label, "model": "RSF", "seed": None, **run_rsf(d)}]
    for s in seeds:
        m, _, _ = evaluate_vadesc(train_vadesc(d, seed=s, epochs=EPOCHS), d)
        rows.append({"split": split, "data": label, "model": "VaDeSC", "seed": s,
                     "C-index": m["C-index"], "IBS": m["IBS"]})
    return rows

rows = []
for split in SPLITS:
    d_median = load_support(drop_post_baseline=True, seed=split)
    d_vae = load_support_from_frame(vae_impute_support(drop_post_baseline=True, split_seed=split, verbose=True),
                                    drop_post_baseline=True, seed=split)
    rows += run_models(d_median, VADESC_SEEDS, split, "median imputation")
    rows += run_models(d_vae, VADESC_SEEDS, split, "VAE imputation")
    print(f"split {split} done", flush=True)
imp = pd.DataFrame(rows)

VAE (Encoder Indicators, split 42): 134 epochs, best valid ELBO -27.024


split 42 done


VAE (Encoder Indicators, split 1): 185 epochs, best valid ELBO -26.002


split 1 done


VAE (Encoder Indicators, split 2): 162 epochs, best valid ELBO -26.384


split 2 done


In [3]:
# mean +/- sd over all runs (splits x seeds for VaDeSC, splits for Cox and RSF)
imp.groupby(["model", "data"])[["C-index", "IBS"]].agg(["mean", "std", "count"]).round(3)

C-index                 IBS             
                            mean    std count   mean    std count
model  data                                                      
Cox PH VAE imputation      0.723  0.009     3  0.178  0.005     3
       median imputation   0.723  0.009     3  0.179  0.004     3
RSF    VAE imputation      0.716  0.006     3  0.177  0.004     3
       median imputation   0.717  0.006     3  0.176  0.004     3
VaDeSC VAE imputation      0.669  0.016    15  0.204  0.006    15
       median imputation   0.653  0.013    15  0.208  0.007    15

In [4]:
# per split (VaDeSC averaged over seeds)
imp.groupby(["model", "split", "data"])["C-index"].mean().unstack("data").round(3)

data          VAE imputation  median imputation
model  split                                   
Cox PH 1               0.722              0.723
       2               0.714              0.714
       42              0.733              0.732
RSF    1               0.714              0.715
       2               0.711              0.712
       42              0.722              0.724
VaDeSC 1               0.655              0.643
       2               0.669              0.653
       42              0.683              0.662

## 2. Is the VaDeSC difference real?

Paired differences (VAE − median) for the same split and the same VaDeSC seed. A 95% confidence interval that excludes 0 means the difference is unlikely to be seed or split noise.

The runs within a split share the same test set, so they are not fully independent; the per-split means are the more conservative check.

In [5]:
def paired(model):
    d = imp[imp.model == model].pivot_table(index=["split", "seed"] if model == "VaDeSC" else ["split"],
                                            columns="data", values="C-index")
    diff = d["VAE imputation"] - d["median imputation"]
    n = len(diff)
    ci = stats.t.ppf(0.975, n - 1) * diff.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    p = stats.ttest_1samp(diff, 0).pvalue if n > 1 else np.nan
    return {"model": model, "pairs": n, "mean diff (VAE - median)": diff.mean(),
            "95% CI": f"[{diff.mean() - ci:.3f}, {diff.mean() + ci:.3f}]", "p-value": p}

pd.DataFrame([paired(m) for m in ["VaDeSC", "Cox PH", "RSF"]]).round(3)

,model,pairs,mean diff (VAE - median),95% CI,p-value
0,VaDeSC,15,0.016,"[0.010, 0.023]",0.000
1,Cox PH,3,0.000,"[-0.001, 0.002]",0.543
2,RSF,3,-0.001,"[-0.002, -0.000]",0.029


In [6]:
# conservative version: one VaDeSC value per split (mean over seeds)
per_split = imp[imp.model == "VaDeSC"].groupby(["split", "data"])["C-index"].mean().unstack("data")
diff = per_split["VAE imputation"] - per_split["median imputation"]
print(per_split.round(3))
print(f"\nper-split differences: {diff.round(3).tolist()} | mean {diff.mean():.3f}")

data   VAE imputation  median imputation
split                                   
1               0.655              0.643
2               0.669              0.653
42              0.683              0.662

per-split differences: [0.012, 0.015, 0.021] | mean 0.016


## 3. Feature leakage (median imputation)

The actual leakage comparison: the same models with and without the post-baseline variables (`sfdm2`, `slos`, `charges`, `totcst`). The "without" results are reused from section 1.

In [7]:
rows = []
for split in SPLITS:
    d_leak = load_support(drop_post_baseline=False, seed=split)
    rows += run_models(d_leak, LEAK_SEEDS, split, "with post-baseline variables")
leak = pd.concat([pd.DataFrame(rows),
                  imp[(imp.data == "median imputation") & (imp.seed.isna() | imp.seed.isin(LEAK_SEEDS))]
                  .assign(data="without post-baseline variables")])
leak.groupby(["model", "data"])["C-index"].agg(["mean", "std", "count"]).unstack("data").round(3)

mean                                  \
data   with post-baseline variables without post-baseline variables   
model                                                                 
Cox PH                        0.800                           0.723   
RSF                           0.804                           0.717   
VaDeSC                        0.769                           0.657   

                                std                                  \
data   with post-baseline variables without post-baseline variables   
model                                                                 
Cox PH                        0.005                           0.009   
RSF                           0.002                           0.006   
VaDeSC                        0.011                           0.015   

                              count                                  
data   with post-baseline variables without post-baseline variables  
model                                                                
Cox PH                            3                               3  
RSF                               3                               3  
VaDeSC                            9                               9

## Notes

- The median-imputed data uses `src.data.load_support` and the VAE-imputed data the same function on the VAE-filled file, so the preprocessing (encoding, scaling, split) is identical; only the filled-in values differ.
- RSF is seeded, Cox is deterministic, so for them the spread comes only from the splits.
- Not covered here: the other VAE variants (Zero Imputation, Encoder Decoder Indicators), several VAE seeds per split, and VAE imputation with the post-baseline variables included.